# 2.1. 데이터 읽기

- 모든 학습자의 공통적 편의성을 위해 데이터 폴더 경로는 'drive/MyDrive/RecoSys/Data' 다음으로 통일한다.
- MoviesLens 100K 데이터는 3가지 파일로 구성
  1. 사용자 데이터 : u.user
  2. 영화에 대한 데이터 : u.item
  3. 영화 평가 : u.data
  

In [1]:
import os
import pandas as pd
from pathlib import Path

base_src = Path.cwd().parent / 'data'
u_user_src = os.path.join(base_src, "u.user")

u_cols = ['user_id', 'age', 'sex', 'occupation', 'zip_code']
users = pd.read_csv(u_user_src, sep='|', names=u_cols, encoding='latin-1')
users = users.set_index('user_id')

users.head()

,age,sex,occupation,zip_code
user_id,,,,
1,24,M,technician,85711
2,53,F,other,94043
3,23,M,writer,32067
4,24,M,technician,43537
5,33,F,other,15213


In [2]:
u_item_src = os.path.join(base_src,'u.item')
i_cols = ['movie_id','title','release date','video release date',
          'IMDB URL','unknown','Action','Adventure','Animation',
          'Children\'s','Comedy','Crime','Documentary','Drama','Fantasy',
          'Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
movies = pd.read_csv(u_item_src,
	    sep='|',
            names=i_cols,
            encoding='latin-1')
movies = movies.set_index('movie_id')
movies.head()

,title,release date,video release date,IMDB URL,unknown,Action,Adventure,Animation,Children's,Comedy,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movie_id,,,,,,,,,,,,,,,,,,,,,
1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [3]:
# u.data 파일을 DataFrame으로 읽기
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')
ratings = ratings.set_index('user_id')
ratings.head()

,movie_id,rating,timestamp
user_id,,,
196,242,3,881250949
186,302,3,891717742
22,377,1,878887116
244,51,2,880606923
166,346,1,886397596


# 2.2. 인기제품 추천 방식

가장 간단한 추천 방식. 
가장 인기 있는 제품을 사용자에게 추천.

구현: 영화 평점 데이터 중 상위 N개를 뽑아서 추천.

In [4]:
def recom_movie(n_items):
  movie_rating_mean = ratings.groupby(['movie_id'])['rating'].mean()
  sorted_movie_rating = movie_rating_mean.sort_values(ascending=False)[:n_items]
  recom_movies = movies.loc[sorted_movie_rating.index]
  recommendation = recom_movies['title']
  return recommendation


recom_movie(5)

movie_id
814                         Great Day in Harlem, A (1994)
1599                        Someone Else's America (1995)
1201           Marlene Dietrich: Shadow and Light (1996) 
1122                       They Made Me a Criminal (1939)
1653    Entertaining Angels: The Dorothy Day Story (1996)
Name: title, dtype: object

# 2.3. 추천 시스템 정확도 측정

**RMSE (Root Mean Square Error) 방식** 

모델의 예측값과 실제값의 차이를 제곱 → 평균 → 제곱근 한 값.

즉, “내가 예측한 평점이 실제 평점에서 평균적으로 얼마나 틀렸는가” 를 숫자로 나타낸 지표.

$$
\mathrm{RMSE}
=
\sqrt{
  \frac{1}{N}
  \sum_{i=1}^{N}
  (\hat{y}_i - y_i)^2
}
$$

In [5]:
# 영화 평점에 대해서 실제값과 best-seller 방식으로 구한 예측값의 RMSE를 계산하는 코드

import numpy as np

def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

# 정확도 계산
rmse = []
movie_mean = ratings.groupby(['movie_id'])['rating'].mean()

for user in set(ratings.index):
    y_true = ratings.loc[user]['rating']
    # best-seller 방식으로
    y_pred = movie_mean[ratings.loc[user]['movie_id']]
    accuracy = RMSE(y_true, y_pred)
    rmse.append(accuracy)

# RMSE 계산
print(np.mean(rmse))

0.996007224010567


# 2.4. 사용자 집단별 추천

In [6]:
# 데이터 읽어오기(user, item, data)
import os
import pandas as pd
base_src = Path.cwd().parent / 'data'
u_user_src = os.path.join(base_src,'u.user')
u_cols = ['user_id','age','sex','occupation','zip_code']
users = pd.read_csv(u_user_src,
	sep = '|',
        names = u_cols,
        encoding = 'latin-1')
u_item_src = os.path.join(base_src,'u.item')
i_cols = ['movie_id','title','release date','video release date','IMDB URL','unknown','Action','Adventure','Animation','Children\'s','Comedy','Crime','Documentary','Drama','Fantasy','Film-Noir','Horror','Musical','Mystery','Romance','Sci-Fi','Thriller','War','Western']
movies = pd.read_csv(u_item_src,
	    sep='|',
            names=i_cols,
            encoding='latin-1')
u_data_src = os.path.join(base_src,'u.data')
r_cols = ['user_id','movie_id','rating','timestamp']
ratings = pd.read_csv(u_data_src,
        sep = '\t',
        names = r_cols,
        encoding='latin-1')


# ratings DataFrame에서 timestamp 제거
ratings = ratings.drop('timestamp',axis=1)
movies = movies[['movie_id','title']]


In [7]:
# 데이터 train, test-set 분리
from sklearn.model_selection import train_test_split

x = ratings.copy()
y = ratings['user_id']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y)

# 정확도 (RMSE) 계산하는 함수
def RMSE(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred))**2))

# 모델별 RMSE 계산하는 함수
def score(model):
    id_pairs = zip(x_test['user_id'], x_test['movie_id'])
    y_pred = np.array([model(user, movie) for (user, movie) in id_pairs])
    y_true = np.array(x_test['rating'])
    return RMSE(y_true, y_pred)

# best-seller 함수를 이용한 정확도 계산
train_mean = x_train.groupby(['movie_id'])['rating'].mean()

def best_seller_model(user_id, movie_id):
    try:
        return train_mean[movie_id]
    except:
        return 3.0

score(best_seller_model)



np.float64(1.0373827467133503)

In [ ]:
# 성별에 따른 예측값 계산
merged_ratings = pd.merge(x_train, users)

users = users.set_index('user_id')

g_mean = merged_ratings[['movie_id', 'sex', 'rating']].groupby(['movie_id', 'sex'])['rating'].mean()

g_mean.head()

rating_matrix = x_train.pivot(index='user_id', columns='movie_id', values='rating')

rating_matrix.head()



movie_id,1,2,3,4,5,6,7,8,9,10,...,1672,1673,1674,1675,1676,1677,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,3.0,3.0,NaN,4.0,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Gender 기준 추천

def cf_gender_model(user_id, movie_id):
    if movie_id in rating_matrix.columns:
        gender = users.loc[user_id]['sex']
        if gender in g_mean[movie_id].index:
            return g_mean[movie_id][gender]
        else:
            return 3.0
    else:
        return 3.0

score(cf_gender_model)


np.float64(1.047965115433832)